# はじめに

このノートブックでは、実際に動作させながら暗号に関する基礎を学びます。

# 準備

下のセルを選択してからメニューの<b>「▶︎| Run」</b>を押して、次のセルを評価してください。<br>
(以降では、「評価」すると呼称します）

In [ ]:
pip install pycryptodome

# ハッシュ関数とハッシュ値

ハッシュ関数は元のデータの特徴を示す値を計算します。<br>
 元のデータが少しでも変更されるとハッシュ関数の値は異なってきます。<br>
ここでは、自分の名前のハッシュ値を求めてみます。

<img src="hash.png" width= 600>

## メッセージを作成します（自分の名前）

暗号化するメッセージを作成します。<br>
次のセルの「室蘭太郎」の代わりに自分の名前を入れて評価してください。

In [ ]:
name = '室蘭太郎'

## ハッシュ値を計算してみます

SHA256という関数（ハッシュ関数）を使ってハッシュ値を計算します。<br>
次のセルを評価してください。nameのハッシュ値を計算します。

In [ ]:
from Crypto.Hash import SHA256

def sha_sum(txt):
    hash_object = SHA256.new(data=txt.encode())
    print(hash_object.hexdigest())
    
sha_sum(name)

## 別の文字でハッシュ値を計算してみます

試しに次のセルで苗字と名前の間に「スペース」を入れて再度実行してみてください。

In [ ]:
name = '室蘭 太郎'
sha_sum(name)

ハッシュ関数の入力を少しでも変えると、ハッシュ値が変わることがわかります。

# 共通鍵暗号

<img src="sharedKey.png" width=600>

## ①鍵の生成（送信者）

次のセルで、乱数を使って共通鍵を作成します。<br>
（作成した共通鍵は'commonKey.txt'というファイルに記録しています。）

In [ ]:
from Crypto.Random import get_random_bytes

def gen_key():
    k = get_random_bytes(16)
    with open("commonKey.txt", "w") as f:
        f.write(k.hex())
    return k

key = gen_key()
print(key.hex())

## ②暗号化するメッセージの作成（送信者）

次のセルで、暗号化したいメッセージを定義します。<br>
自由にメッセージを変えてみてください。

In [ ]:
message = 'おはようございます'

## ③共通鍵を使った暗号化（送信者）

次のセルで、共通鍵を使ってメッセージを暗号化します。<br>
セルを評価すると、暗号化されたメッセージが表示されます。

In [ ]:
from Crypto.Cipher import AES

def aes_encrypt(key, msg):
    cipher = AES.new(key, AES.MODE_EAX)
    nonce = cipher.nonce
    ciphertext, tag = cipher.encrypt_and_digest(msg.encode('utf-8'))
    return (nonce, ciphertext, tag)

tpl = aes_encrypt(key, message)
tpl[0].hex(),tpl[1].hex(), tpl[2].hex()

## ④共通鍵を使った復号（受信者）

メッセージを受け取ったとみなして、次のセルで共通鍵を使ってメッセージを復号します。<br>
4.2で入れたメッセージと同じ物が出れば正解です。

In [ ]:
from Crypto.Cipher import AES

def aes_decrypt(key, tpl):
    nonce, ciphertext, tag = tpl
    cipher = AES.new(key, AES.MODE_EAX, nonce=nonce)
    plaintext = cipher.decrypt(ciphertext)
    try:
        cipher.verify(tag)
        print("メッセージ本物です:", plaintext.decode('utf-8'))
    except ValueError:
        print("共通鍵がちがう、またはメッセージが壊れています")
    return plaintext.decode('utf-8')

decrrypted_message = aes_decrypt(key, tpl)
decrrypted_message

# 公開鍵暗号

<img src="publicKey.png" width=600>

## ①公開鍵と秘密鍵の生成（受信者）

乱数を使って、RSAの秘密鍵と公開鍵を作成します。<br>

1. 秘密鍵：private_key
2. 公開鍵：public_key

また、次のファイルが作成されます。

1. 秘密鍵：private.pem

2. 公開鍵：public.pem

In [ ]:
from Crypto.PublicKey import RSA

# 秘密鍵の生成
private_key = RSA.generate(2048)
with open("private.pem", "w") as f:
  tmp = private_key.export_key().decode('utf-8')
  print(tmp)
  f.write(tmp)

# 秘密鍵から公開鍵を生成
public_key = private_key.publickey()
with open("public.pem", "w") as f:
  tmp = public_key.export_key().decode('utf-8')
  print(tmp)
  f.write(tmp)

## ②暗号化するメッセージを作成する（送信者）

次のセルで、暗号化したいメッセージを定義します。<br>
自由にメッセージを変えてみてください。

In [ ]:
message = 'こんにちは'

## ③公開鍵を使った暗号化（送信者）

公開鍵を使って、文字を暗号化します。

In [ ]:
from Crypto.PublicKey import RSA
from Crypto.Cipher import PKCS1_OAEP

def encrypt_ras(pubkey, txt):
    cipher_rsa = PKCS1_OAEP.new(pubkey)
    ctext = cipher_rsa.encrypt(txt.encode())
    print('暗号化されたメッセージ：', ctext.hex())
    return ctext

ciphertext = encrypt_ras(public_key, message)

## ④秘密鍵を使った復号（受信者）

秘密鍵を使って復号します。

In [ ]:
def decrypt_rsa(prikey, ctext):
    decipher_rsa = PKCS1_OAEP.new(prikey)
    msg = decipher_rsa.decrypt(ctext).decode("utf-8")
    return msg

decrypt_rsa(private_key, ciphertext)